# MF-HGNN MDD_SiteLeaveGroupOut Reproduction Guide

This notebook includes:

1. Data loading and a brief overview
2. Dataset splitting (site-wise leave-group-out cross-validation) and summary
3. Full training procedure (calling `main.py`)
4. Testing with the best saved models and displaying test logs

All core training and evaluation logic is identical to the original code used in the paper, implemented via `main.py`, `dataload.py`, `model.py`, etc.


In [1]:
# Environment and dependencies

import os
import numpy as np
import torch

from opt import OptInit
from dataload import dataloader

# Initialize hyperparameters (same as running main.py from command line)
opt = OptInit().initialize()
print('Device:', opt.device)

# Explicitly print several important paths for this notebook
print('subject_IDs_path:', opt.subject_IDs_path)
print('phenotype_path:', opt.phenotype_path)
print('data_path:', opt.data_path)
print('log_path:', opt.log_path)
print('ckpt_path:', opt.ckpt_path)


 Using GPU in torch
==========       CONFIG      =============
train:1
use_cpu:False
lr:0.01
wd:5e-05
num_iter:400
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./
log_path:./inffus_log.txt
subject_IDs_path:/mnt/restmdd_ho_841_2/MDD_pcp/subject_IDs.txt
phenotype_path:/mnt/restmdd_ho_841_2/MDD_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/restmdd_ho_841_2/MDD_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is train.
 Using GPU in torch
==========       CONFIG      =============
train:1
use_cpu:False
lr:0.01
wd:5e-05
num_iter:400
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./
log_path:./inffus_log.txt
subject_IDs_path:/mnt/restmdd_ho_841_2/MDD_pcp/subject_IDs.txt
phenotype_path:/mnt/restmdd_ho_841_2/MDD_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/restmdd_ho_841_2/MDD_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END   

## 1. Data loading and overview

In this section we directly use the `dataloader` class from `dataload.py` to perform exactly the same data-loading pipeline as in the official experiments, and briefly inspect key data structures:

- Raw node features `raw_features`
- Label vector `y`
- Phenotypic data `phonetic_data` (site, sex, age, etc.)
- Dynamic functional connectivity `dynamic_fc`


In [2]:
# 1. Data loading (with detailed prints)

dl = dataloader()
raw_features, y, nonimg, phonetic_score, time_series, dynamic_fc = dl.load_data()

# Avoid flooding the notebook output; only show parts of large arrays
np.set_printoptions(threshold=20)

# Basic information
print('Number of raw_features samples:', len(raw_features))
print('Shape of labels y:', np.array(y).shape)
print('Shape of non-image phenotypes nonimg:', np.array(nonimg).shape)
print('Number of dynamic_fc samples:', len(dynamic_fc))

# Inspect the first sample
first_feat = raw_features[0]
print('\nType of the first sample:', type(first_feat))
if hasattr(first_feat, 'shape'):
    print('Shape of the first sample features:', first_feat.shape)
elif hasattr(first_feat, 'x'):
    # For torch_geometric.data.Data
    print('Shape of node features x of the first sample:', first_feat.x.shape)
    print('Shape of edge_index of the first sample:', first_feat.edge_index.shape)

# Label distribution
unique, counts = np.unique(y, return_counts=True)
print('\nLabel distribution (label: count):')
for u, c in zip(unique, counts):
    print(f'  {u}: {c}')

# Phenotypic information (SITE_ID, SEX, AGE_AT_SCAN)
print('\nSITE_ID for all samples:')
print(phonetic_score['SITE_ID'])

print('\nSEX for all samples:')
print(phonetic_score['SEX'])

print('\nAGE_AT_SCAN for all samples:')
print(phonetic_score['AGE_AT_SCAN'])


Number of raw_features samples: 841
Shape of labels y: (841,)
Shape of non-image phenotypes nonimg: (841, 3)
Number of dynamic_fc samples: 841

Type of the first sample: <class 'torch_geometric.data.data.Data'>
Shape of node features x of the first sample: torch.Size([112, 112])
Shape of edge_index of the first sample: torch.Size([2, 504])

Label distribution (label: count):
  0.0: 457
  1.0: 384

SITE_ID for all samples:
[0. 0. 0. ... 2. 2. 2.]

SEX for all samples:
[1. 2. 1. ... 2. 1. 1.]

AGE_AT_SCAN for all samples:
[26. 26. 43. ... 74. 78. 63.]


## 2. Dataset splitting strategy in the current code

In this repository, dataset splitting is implemented in `dataload.py` via `dataloader.data_split()`. The **current implementation** is a strict site-wise leave-one-site-out cross-validation. The logic is:

- In `dataloader.data_split(self, n_folds)`:
  - Extract the integer-encoded site ID for each subject.
  - Compute the set of unique site IDs.
  - For each site ID `site_id`:
    - All samples from this site form the **test set**.
    - All samples from the remaining sites form the **training set**.


In [3]:
# 2*. Detailed illustration of the site-wise leave-group-out splitting

from dataload import get_ids, get_subject_score

# Use the same opt and dl as above
cv_splits = dl.data_split(opt.n_folds)
n_folds = len(cv_splits)

labels = np.array(y)
sites = np.array(phonetic_score['SITE_ID'])  # integer-encoded site IDs from load_data

# Recover mapping "site index -> original SITE_ID string" according to dataload.py
subject_list = get_ids()
site_dict = get_subject_score(subject_list, score='SITE_ID')  # {subject_id: SITE_ID string}
unique_site_names = np.unique(list(site_dict.values())).tolist()
site_id_to_name = {idx: name for idx, name in enumerate(unique_site_names)}

print(f'Total number of samples: {len(labels)}')
print(f'Number of sites / effective folds n_folds: {n_folds}')
print('Mapping from site index to original SITE_ID:')
for sid, name in site_id_to_name.items():
    print(f'  {sid}: {name}')
print('\n')

for fold, (train_idx, test_idx) in enumerate(cv_splits):
    y_tr, s_tr = labels[train_idx], sites[train_idx]
    y_te, s_te = labels[test_idx], sites[test_idx]

    u_te_site, c_te_site = np.unique(s_te, return_counts=True)

    print(f'====== Fold {fold} ======')
    print(f'Number of training samples: {len(train_idx)}, number of test samples: {len(test_idx)}')

    # For each fold, the test set should contain samples from a single site
    print('Test set site distribution (SITE_ID index -> count):')
    print(dict(zip(u_te_site.astype(int), c_te_site)))

    # Corresponding original SITE_ID names
    test_site_name_counts = {}
    for sid, cnt in zip(u_te_site.astype(int), c_te_site):
        site_name = site_id_to_name.get(sid, f'UNKNOWN_{sid}')
        test_site_name_counts[site_name] = int(cnt)
    print('Test set site name distribution (SITE_NAME -> count):')
    print(test_site_name_counts)

    print()


Total number of samples: 841
Number of sites / effective folds n_folds: 3
Mapping from site index to original SITE_ID:
  0: S20
  1: S21
  2: S25


====== Fold 0 ======
Number of training samples: 308, number of test samples: 533
Test set site distribution (SITE_ID index -> count):
{0: 533}
Test set site name distribution (SITE_NAME -> count):
{'S20': 533}

====== Fold 1 ======
Number of training samples: 685, number of test samples: 156
Test set site distribution (SITE_ID index -> count):
{1: 156}
Test set site name distribution (SITE_NAME -> count):
{'S21': 156}

====== Fold 2 ======
Number of training samples: 689, number of test samples: 152
Test set site distribution (SITE_ID index -> count):
{2: 152}
Test set site name distribution (SITE_NAME -> count):
{'S25': 152}



## 3. Training example (calling main.py)

The official training and evaluation pipeline is implemented in `main.py` under `if __name__ == "__main__":`, which includes:

- Initializing hyperparameters (`OptInit`)
- Calling `dataloader` to load the data
- Running K-fold cross-validation
- Printing training and test metrics at each epoch
- Saving the best model (on the test set) for each fold to `opt.ckpt_path`

In this notebook we invoke `main.py` via a command-line call so that the behavior is exactly the same as running the script directly, and a **training log file** is automatically generated (handled by the `Logger` class).

> Training from scratch can be time-consuming, so in practice we usually pre-train on the server, save the best models and logs, and then use this notebook mainly for demonstration.


In [ ]:
import sys

# Use the Python interpreter of the current notebook kernel
!{sys.executable} main.py --train 1 --num_iter 500 --n_folds 10 --ckpt_path ./ckpt_demo/ --log_path ./run.log


In [4]:
# 3.2 View a snippet of the training log

log_path = './run.log'

if os.path.exists(log_path):
    print(f'Reading log file: {log_path}\n')
    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Show only the last 40 lines for readability
    for line in lines[-40:]:
        print(line.rstrip())
else:
    print('Log file does not exist. Please run the training cell above first.')


Reading log file: ./run.log

Epoch: 372,	ce loss: 0.17123,	ce loss_cla: 0.17123,	train acc: 0.94194,	test acc: 0.67105,	test spe: 0.43820,	test sen: 1.00000      2026-03-25 11:45:29
Epoch: 373,	ce loss: 0.05368,	ce loss_cla: 0.05368,	train acc: 0.98403,	test acc: 0.68421,	test spe: 0.46067,	test sen: 1.00000      2026-03-25 11:46:28
Epoch: 374,	ce loss: 0.06581,	ce loss_cla: 0.06581,	train acc: 0.97678,	test acc: 0.73026,	test spe: 0.53933,	test sen: 1.00000      2026-03-25 11:47:26
Epoch: 375,	ce loss: 0.07035,	ce loss_cla: 0.07035,	train acc: 0.97823,	test acc: 0.75000,	test spe: 0.57303,	test sen: 1.00000      2026-03-25 11:48:26
Epoch: 376,	ce loss: 0.07938,	ce loss_cla: 0.07938,	train acc: 0.96226,	test acc: 0.73684,	test spe: 0.56180,	test sen: 0.98413      2026-03-25 11:49:25
Epoch: 377,	ce loss: 0.06054,	ce loss_cla: 0.06054,	train acc: 0.97388,	test acc: 0.72368,	test spe: 0.52809,	test sen: 1.00000      2026-03-25 11:50:24
Epoch: 378,	ce loss: 0.06743,	ce loss_cla: 0.06743,	t

## 4. Evaluation with the best saved models

In `main.py`, the `evaluate()` function is responsible for:

- Loading the best-performing model for each fold from `opt.ckpt_path`.
- Running forward passes on the corresponding test set.
- Computing and printing multiple metrics (accuracy, AUC, sensitivity, specificity, F1-score, etc.).

Here we call the same script with `--train` set to `0` to perform evaluation using the saved models.

> Before evaluation, make sure that the corresponding model checkpoints for all folds have been successfully saved in `ckpt_path` during training.


In [5]:
import sys

# 4.1 Example: evaluate using pre-trained models

# Assume the training stage used ./ckpt_demo/ as ckpt_path.
# Here we keep the same ckpt_path, set train to 0, and run evaluation only.

!{sys.executable} main.py --train 0 --num_iter 400 --n_folds 10 --ckpt_path ./ckpt_demo/ --log_path ./test.log


 Using GPU in torch
==========       CONFIG      =============
train:0
use_cpu:False
lr:0.01
wd:5e-05
num_iter:400
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./ckpt_demo/
log_path:./test.log
subject_IDs_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is eval.
 Using GPU in torch
==========       CONFIG      =============
train:0
use_cpu:False
lr:0.01
wd:5e-05
num_iter:400
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./ckpt_demo/
log_path:./test.log
subject_IDs_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
=========

In [6]:
# 4.2 View a snippet of the evaluation log

log_path_test = './test.log'

if os.path.exists(log_path_test):
    print(f'Reading evaluation log file: {log_path_test}\n')
    with open(log_path_test, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Show the last 40 lines of evaluation results
    for line in lines[-40:]:
        print(line.rstrip())
else:
    print('Evaluation log file does not exist. Please run the evaluation cell above first.')


Reading evaluation log file: ./test.log

    (convs4): ModuleList(
      (0): TransformerConv(2020, 20, heads=1)
      (1-4): 4 x TransformerConv(40, 20, heads=1)
    )
    (bns): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bns2): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bns3): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (out_fc): Linear(in_features=100, out_features=2, bias=True)
    (conv5): Linear(in_features=20, out_features=1, bias=False)
    (conv6): Linear(in_features=20, out_features=1, bias=False)
    (conv7): Linear(in_features=20, out_features=1, bias=False)
    (conv8): Linear(in_features=20, out_features=1, bias=False)
    (conv9): Linear(in_features=20, out_features=1, bias=False)
    (conv10): Linear(in_features=20, out_features=1, bias=False)
 